**Voice-to-Grasp - additional statistics cells** for `MultiUser_CombinedAnalysis_v3.ipynb` (effect sizes).
Note: with four participants these are descriptive only (paper Section 2.9).

# Statistical Analysis Cells — MultiUser_CombinedAnalysis_v3.ipynb
Add these cells **after Cell 3 (Results Table + Wilcoxon)** in your existing notebook.
They all use the `USERS` list already loaded in Cell 2.

---
## STAT CELL A · Effect Size (Cohen's d) — Baseline vs Personalised
Your existing Cell 3 already runs the Wilcoxon test. This cell adds
Cohen's d effect size, which reviewers increasingly require alongside p-values.

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

baseline_accs = [u["baseline_acc"] for u in USERS]
personal_accs = [u["test_acc"]     for u in USERS]
gains         = [p - b for p, b in zip(personal_accs, baseline_accs)]

# Cohen's d for paired data
mean_diff = np.mean(gains)
std_diff  = np.std(gains, ddof=1)
cohen_d   = mean_diff / std_diff if std_diff > 0 else float('inf')

if   abs(cohen_d) >= 0.8: d_interp = "large"
elif abs(cohen_d) >= 0.5: d_interp = "medium"
elif abs(cohen_d) >= 0.2: d_interp = "small"
else:                     d_interp = "negligible"

# Wilcoxon (rerun cleanly for reporting)
stat_w, p_w = wilcoxon(personal_accs, baseline_accs, zero_method="zsplit")

print("=" * 60)
print("Personalisation Effect: Baseline vs Personalised Accuracy")
print("=" * 60)
print(f"  N participants          : {len(USERS)}")
print(f"  Mean baseline accuracy  : {np.mean(baseline_accs):.4f} (SD={np.std(baseline_accs):.4f})")
print(f"  Mean personalised acc   : {np.mean(personal_accs):.4f} (SD={np.std(personal_accs):.4f})")
print(f"  Mean gain               : {mean_diff:+.4f} (SD={std_diff:.4f})")
print(f"")
print(f"  Wilcoxon W              : {stat_w:.4f}")
print(f"  p-value                 : {p_w:.4f}")
sig = 'SIGNIFICANT' if p_w < 0.05 else 'not significant'
print(f"  -> {sig} at alpha = 0.05")
print(f"")
print(f"  Cohen's d (effect size) : {cohen_d:.4f} ({d_interp} effect)")
print("=" * 60)

---
## STAT CELL B · Bootstrap Confidence Intervals — Mean Accuracy Gain
Since your sample is small (n=5), bootstrap CIs are more reliable
than parametric CIs. This gives you a robust interval to report
alongside your Wilcoxon result.

In [ ]:
import numpy as np

np.random.seed(42)
gains = np.array([u["test_acc"] - u["baseline_acc"] for u in USERS])

n_boot = 10000
boot_means = [
    np.mean(np.random.choice(gains, size=len(gains), replace=True))
    for _ in range(n_boot)
]

ci_lo = np.percentile(boot_means, 2.5)
ci_hi = np.percentile(boot_means, 97.5)
observed_mean = np.mean(gains)

# One-sided p-value: proportion of bootstrap samples with gain <= 0
p_boot = np.mean(np.array(boot_means) <= 0)

print("=" * 60)
print(f"Bootstrap Analysis of Mean Accuracy Gain (n_boot={n_boot})")
print("=" * 60)
print(f"  Observed mean gain      : {observed_mean:+.4f}")
print(f"  95% Bootstrap CI        : [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"  Bootstrap p-value       : {p_boot:.4f} (one-sided: gain <= 0)")
sig = 'SIGNIFICANT' if p_boot < 0.05 else 'not significant'
print(f"  -> {sig} at alpha = 0.05")
print("")
print(f"  Per-user gains: {[round(g, 4) for g in gains]}")
print("=" * 60)

---
## STAT CELL C · Per-Class F1 Improvement — Wilcoxon Test Per Category
Tests whether the F1 improvement for each grasp category is statistically
significant across participants. Applies Bonferroni correction for
multiple comparisons (7 categories).

In [ ]:
from scipy.stats import wilcoxon
import numpy as np

alpha      = 0.05
n_tests    = len(LABEL_ORDER)
alpha_bonf = alpha / n_tests  # Bonferroni correction

print("=" * 75)
print("Per-Class F1 Improvement: Wilcoxon Test (with Bonferroni Correction)")
print(f"  Alpha = {alpha}, Bonferroni-corrected alpha = {alpha_bonf:.4f}")
print("=" * 75)

fmt = "{:<16} {:>10} {:>10} {:>10} {:>10} {:>12}"
print(fmt.format("Grasp Class", "Base F1", "Pers F1", "Mean Gain", "p-value", "Significant"))
print("-" * 75)

for label in LABEL_ORDER:
    base_f1s = [u["f1_base"][label] for u in USERS]
    pers_f1s = [u["f1_pers"][label] for u in USERS]
    gain     = np.mean(pers_f1s) - np.mean(base_f1s)

    try:
        stat, p = wilcoxon(pers_f1s, base_f1s, zero_method="zsplit")
        sig = "YES *" if p < alpha_bonf else ("yes (uncorr)" if p < alpha else "no")
    except Exception:
        p   = float("nan")
        sig = "n/a (no variance)"

    print(fmt.format(
        LABEL_DISPLAY.get(label, label),
        f"{np.mean(base_f1s):.3f}",
        f"{np.mean(pers_f1s):.3f}",
        f"{gain:+.3f}",
        f"{p:.4f}" if not np.isnan(p) else "  n/a",
        sig
    ))

print("-" * 75)
print("  * Significant after Bonferroni correction")
print("  yes (uncorr) = significant without but not after Bonferroni correction")

---
## STAT CELL D · 95% Confidence Intervals — Baseline and Personalised Accuracy
Reports Wilson CIs for each participant's baseline and personalised accuracy.
These can be added directly to your results table in the paper.

In [ ]:
from statsmodels.stats.proportion import proportion_confint
import numpy as np

# Test phase has 70 prompts (10 per category x 7 categories)
# Adaptation phase has 140 prompts
N_TEST  = 70
N_TRAIN = 140

print("=" * 80)
print("95% Wilson Confidence Intervals — Baseline and Personalised Accuracy")
print("=" * 80)

fmt = "{:<12} {:>12} {:>20} {:>12} {:>20}"
print(fmt.format("Participant", "Base Acc", "Base 95% CI", "Pers Acc", "Pers 95% CI"))
print("-" * 80)

for u in USERS:
    # Baseline: from training phase
    b_correct = round(u["baseline_acc"] * N_TRAIN)
    b_lo, b_hi = proportion_confint(b_correct, N_TRAIN, method="wilson")

    # Personalised: from test phase
    p_correct = round(u["test_acc"] * N_TEST)
    p_lo, p_hi = proportion_confint(p_correct, N_TEST, method="wilson")

    print(fmt.format(
        u["name"],
        f"{u['baseline_acc']:.3f}",
        f"[{b_lo:.3f}, {b_hi:.3f}]",
        f"{u['test_acc']:.3f}",
        f"[{p_lo:.3f}, {p_hi:.3f}]"
    ))

print("-" * 80)
mean_base = np.mean([u["baseline_acc"] for u in USERS])
mean_pers = np.mean([u["test_acc"]     for u in USERS])
print(fmt.format("Mean", f"{mean_base:.3f}", "", f"{mean_pers:.3f}", ""))
print("=" * 80)

---
## STAT CELL E · Update Count vs Personalisation Gain — Spearman Correlation
Formally quantifies the relationship between number of model updates
and accuracy gain shown in your scatter plot (Figure 6).

In [ ]:
from scipy.stats import spearmanr, pearsonr
import numpy as np

updates = np.array([u["updates"] for u in USERS])
gains   = np.array([u["test_acc"] - u["baseline_acc"] for u in USERS])

rho, p_spear  = spearmanr(updates, gains)
r,   p_pears  = pearsonr(updates,  gains)

print("=" * 60)
print("Correlation: Model Updates vs Personalisation Gain")
print("=" * 60)
print(f"  Updates  : {list(updates)}")
print(f"  Gains    : {[round(g,4) for g in gains]}")
print("")
print(f"  Spearman rho = {rho:.4f},  p = {p_spear:.4f}")
print(f"  Pearson  r   = {r:.4f},    p = {p_pears:.4f}")
print("")
sig_s = 'SIGNIFICANT' if p_spear < 0.05 else 'not significant'
sig_p = 'SIGNIFICANT' if p_pears < 0.05 else 'not significant'
print(f"  Spearman -> {sig_s} at alpha = 0.05")
print(f"  Pearson  -> {sig_p} at alpha = 0.05")
print("")
print("  NOTE: n=5 is small; use Spearman as primary (non-parametric).")
print("=" * 60)